In [9]:
import os
os.chdir("/home/jovyan/work/repo_conteo_gallinas")
os.environ["CUDA_VISIBLE_DEVICES"] = "0, 2"

import torch
from ultralytics import YOLO

import time
import pandas as pd

from siuba import *

import shutil
from itertools import product


In [10]:
def evaluar_modelo(MODEL_EPOCHS, MODEL_BATCH, MODEL_LR, MODEL_OPTIMIZER):

    # Eliminar caarpetas de resultados
    if os.path.exists("runs"):
        shutil.rmtree("runs")

    # Modelo
    model = YOLO("model/yolo26m.pt")
    
    # Empezar el entrenamiento
    model.train(
        data="/home/jovyan/work/repo_conteo_gallinas/artifacts/full_dataset.yaml",
        imgsz=640,
        epochs=MODEL_EPOCHS,
        batch=MODEL_BATCH,
        lr0=MODEL_LR,
        optimizer=MODEL_OPTIMIZER,
        # device=0,
        device=[0, 1],
        workers=0,
        project='.',
        name='.'
    )

    # Lectura resultados
    df = pd.read_csv("runs/detect/results.csv")
    epoca_final = int(df["epoch"].max())
    tiempo_segundos = float(df["time"].max())
    tiempo_min = tiempo_segundos/60
    tiempo_hora = tiempo_min/60
    best = df.loc[df["metrics/mAP50-95(B)"].idxmax()]
    map50_final = float(best["metrics/mAP50(B)"])
    map5095_final = float(best["metrics/mAP50-95(B)"])
    precision_final = float(best["metrics/precision(B)"])
    recall_final = float(best["metrics/recall(B)"])

    # Número de imágenes
    imagenes = len(os.listdir("data/deteccion/imgs"))
    imagenes_train = len(os.listdir("data/dataset/train/images"))
    imagenes_val = len(os.listdir("data/dataset/val/images"))

    # Información sonsolidada
    data_list = {
        'epocas_inicial': MODEL_EPOCHS
        , 'epocas_fin': epoca_final
        , 'batch': MODEL_BATCH
        , 'learning_rate': MODEL_LR
        , 'optimizer': MODEL_OPTIMIZER
        , 'total_imagenes': imagenes
        , 'total_imagenes_train': imagenes_train
        , 'total_imagenes_val': imagenes_val

        , 'tiempo_minutos': tiempo_min
        , 'tiempo_horas': tiempo_hora
        , 'map50': map50_final
        , 'map50_95': map5095_final
        , "precision": precision_final
        , "recall": recall_final
    }

    return data_list
    



In [ ]:

# ls_epocas = [50]
# ls_batch = [6, 8, 10]
# ls_optimizador = ["SGD"]
# ls_lr = [0.01, 0.001, 0.0001]

ls_epocas = [50, 100, 150]
ls_batch = [6, 8, 10, 12]
ls_optimizador = ["SGD", "AdamW"]
ls_lr = [0.1, 0.01, 0.001]


combinaciones = list(product(ls_epocas, ls_batch, ls_lr, ls_optimizador))
list_epocas, list_batch, list_lr, list_optimizador = zip(*combinaciones)

list_dict_datos = list()
for i, j, k, m in zip(list_epocas, list_batch, list_lr, list_optimizador):

    data_resultado = evaluar_modelo(i, j, k, m)

    list_dict_datos.append(data_resultado)
    

DataResultado11 = pd.DataFrame(list_dict_datos)



Ultralytics 8.4.96 🚀 Python-3.13.11 torch-2.10.0+cu128 CUDA:0 (NVIDIA RTX 6000 Ada Generation, 48640MiB)
                                                       CUDA:1 (NVIDIA RTX 6000 Ada Generation, 48632MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=6, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/jovyan/work/repo_conteo_gallinas/artifacts/full_dataset.yaml, degrees=0.0, deterministic=True, device=0,1, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.1, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, 

In [ ]:

DataResultado12 = (
    DataResultado11
    >> arrange(-_.map50_95)
)
DataResultado12.to_csv("artifacts/data_comparativa_modelos_11.csv", encoding = "utf-8", index = False)



In [ ]:
DataResultado12
